In [1]:
from pathlib import Path
import sys

from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Find the project root whether Jupyter starts from the project or tests/.
project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "tools").is_dir():
    project_root = project_root.parent
if not (project_root / "tools").is_dir():
    raise FileNotFoundError("Could not find the project root containing tools/.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [2]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

response = llm.invoke(
    "Say hello and explain in one sentence what an intelligent vehicle assistant does."
)

print(response.content)

Hello! An intelligent vehicle assistant uses AI to understand driver commands, provide real‑time navigation, safety alerts, and control vehicle functions to make driving safer and more convenient.


In [3]:
from pathlib import Path
import sys

from langchain_core.tools import tool

# Make project packages importable when this notebook runs from tests/.
project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "tools").is_dir():
    project_root = project_root.parent
if not (project_root / "tools").is_dir():
    raise FileNotFoundError("Could not find the project root containing tools/.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tools.weather import get_weather


@tool
def weather_tool(city: str) -> dict:
    """Get the current weather for a city."""
    return get_weather(city)

In [4]:
llm_with_tools = llm.bind_tools([
    weather_tool
])

In [5]:
response = llm_with_tools.invoke(
    "What is the weather in Sfax?"
)

print(response)


content='' additional_kwargs={'reasoning_content': 'User asks: "What is the weather in Sfax?" Need to get current weather. Use function weather_tool with city "Sfax".', 'tool_calls': [{'id': 'fc_aa265e9d-d46a-4c77-83fb-80e62c19f7a3', 'function': {'arguments': '{"city":"Sfax"}', 'name': 'weather_tool'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 129, 'total_tokens': 186, 'completion_time': 0.119782618, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.006114153, 'prompt_tokens_details': None, 'queue_time': 0.049115417, 'total_time': 0.125896771}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_f640395b96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0bfe7-2724-7263-8488-c1098861701d-0' tool_calls=[{'name': 'weather_tool', 'args': {'city': 'Sfax'}, 'id': 'fc_aa265e9d-d46a-4c77-83fb-80e62c19f7a3', 'type': 'tool_call'}] inva

In [6]:
print(response.tool_calls)

[{'name': 'weather_tool', 'args': {'city': 'Sfax'}, 'id': 'fc_aa265e9d-d46a-4c77-83fb-80e62c19f7a3', 'type': 'tool_call'}]


In [7]:
from langchain_core.messages import HumanMessage, ToolMessage
from tools.routing import get_route


@tool
def weather_tool(city: str) -> dict:
    """Get the current weather conditions for a city."""
    return get_weather(city)


@tool
def route_tool(origin: str, destination: str) -> dict:
    """Get driving route information between an origin and a destination."""
    return get_route(origin, destination)

In [8]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

tools = [
    weather_tool,
    route_tool
]

AVAILABLE_TOOLS = {tool.name: tool for tool in tools}

llm_with_tools = llm.bind_tools(tools)

In [9]:
question = "How far is Gabes from Sfax by car?"

messages = [
    HumanMessage(content=question)
]

response = llm_with_tools.invoke(messages)

print("LLM content:")
print(response.content)

print("\nTool calls:")
print(response.tool_calls)

LLM content:


Tool calls:
[{'name': 'route_tool', 'args': {'destination': 'Sfax, Tunisia', 'origin': 'Gabes, Tunisia'}, 'id': 'fc_547b85e2-531d-466a-a89a-2c0a5a1326f8', 'type': 'tool_call'}]


In [10]:
tool_map = {
    "weather_tool": weather_tool,
    "route_tool": route_tool
}

messages.append(response)

for tool_call in response.tool_calls:

    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    selected_tool = tool_map[tool_name]

    result = selected_tool.invoke(tool_args)

    print("Executed tool:", tool_name)
    print("Arguments:", tool_args)
    print("Result:", result)

    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    )

Executed tool: route_tool
Arguments: {'destination': 'Sfax, Tunisia', 'origin': 'Gabes, Tunisia'}
Result: {'origin': {'name': 'Gabès', 'country': 'Tunisia', 'latitude': 33.88146, 'longitude': 10.0982}, 'destination': {'name': 'Sfax', 'country': 'Tunisia', 'latitude': 34.74056, 'longitude': 10.76028}, 'distance_km': 160.12, 'duration_minutes': 111.4, 'uses_highway': True, 'has_toll': True, 'has_ferry': False}


In [11]:
final_response = llm_with_tools.invoke(messages)

print("Final answer:")
print(final_response.content)

Final answer:
By car, the drive from **Gabès** to **Sfax** covers roughly **160 km (about 100 miles)**.  
The typical travel time is around **1 hour 50 minutes** (≈ 111 minutes), assuming normal traffic and using the main highway route.


In [12]:
SYSTEM_PROMPT = """
You are an intelligent vehicle decision-support assistant.

Use the available tools whenever the user asks for real-time or calculated
information such as weather, routes, distance, travel time, tolls, highways,
ferries, or vehicle state.

Before answering:

1. Identify exactly what information the user requested.
2. Select the appropriate tool or tools.
3. Never invent tool results.
4. After receiving tool results, check that all requested fields are present.
5. Convert technical values into driver-friendly information:
   - minutes into hours and minutes
   - weather codes into descriptions
   - booleans into clear Yes/No statements
6. Distinguish between:
   - facts returned by tools
   - reasonable estimates or inferences
7. Mention important uncertainty:
   - routing times may exclude traffic, stops, and ferry waiting
   - weather conditions may change
8. If a tool fails or information is missing, explain that clearly.
9. Give the final answer in concise, natural language.
10. Do not display private chain-of-thought. Provide only a short explanation
    of the important facts and conclusions.
"""

In [13]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(
        content=(
            "I'm currently in Tunis and I want to drive to Houmt Souk in "
            "Djerba. How far is the trip, approximately how long will it "
            "take, and does the route include highways, toll roads, or a ferry?"
        )
    ),
]

response = llm_with_tools.invoke(messages)

In [14]:
from IPython.display import Markdown, display

# Reset to the system and user messages, making this cell safe to rerun.
messages = messages[:2]
messages.append(response)

if not response.tool_calls:
    raise RuntimeError("The LLM did not request a tool for this question.")

for tool_call in response.tool_calls:
    tool_name = tool_call["name"]
    if tool_name not in AVAILABLE_TOOLS:
        raise ValueError(f"Unsupported tool requested: {tool_name}")
    selected_tool = AVAILABLE_TOOLS[tool_name]
    tool_message = selected_tool.invoke(tool_call)
    messages.append(tool_message)
    print(f"Executed: {tool_name}")
    print(f"Tool result: {tool_message.content}\n")

final_response = llm_with_tools.invoke(messages)

display(Markdown("## Final answer"))
display(Markdown(final_response.content))

Executed: route_tool
Tool result: {"origin": {"name": "Tunis", "country": "Tunisia", "latitude": 36.81897, "longitude": 10.16579}, "destination": {"name": "Houmt Souk", "country": "Tunisia", "latitude": 33.87576, "longitude": 10.85745}, "distance_km": 529.81, "duration_minutes": 362.9, "uses_highway": true, "has_toll": true, "has_ferry": true}



## Final answer

- **Distance:** about **530 km** (329 mi).  
- **Estimated driving time:** roughly **6 hours 3 minutes** (362 minutes). This is a best‑case estimate and does not include possible traffic, stops, or waiting time for the ferry.  
- **Route characteristics:**  
  * **Highways:** Yes – the main legs use the A1 motorway.  
  * **Toll roads:** Yes – there are toll sections on the A1.  
  * **Ferry:** Yes – the crossing to Djerba requires a ferry (typically from the port near Mahdia or Sfax).  

So you can expect a ~530 km drive, about six hours on the road, using highways, paying tolls, and taking a ferry to reach Houmt Souk. Safe travels!

In [ ]:
from pprint import pprint

from simulation.trip_simulator import TripSimulator
from tools.vehicle import get_vehicle_state

simulator = TripSimulator(
    origin="Route El Ain, central Sfax",
    destination="North-east central Sfax",
    traffic_level="normal",
)

# Start the SUMO IONIQ 5 simulation.
simulator.start(use_gui=False)

# Advance the vehicle by one simulated minute.
simulator.step(10)

vehicle_state = get_vehicle_state(simulator)

pprint(vehicle_state)

{'health': {'battery_voltage': 12.6,
            'engine_temp_c': 75.0,
            'fuel_percent': 98.63,
            'oil_life_percent': 99.91,
            'tire_pressure_psi': 32.0},
 'motion': {'current_road': 'route_segment_197',
            'elapsed_time_seconds': 990.0,
            'latitude': 34.815722,
            'longitude': 10.734288,
            'speed_kmh': 0.0,
            'travelled_distance_km': 9.14,
            'trip_progress_percent': 100.0},
 'status': 'completed'}
